# Baseline 2 — MLP on Region Features

No sequence modeling. Encodes the prefix using only the first and last region's
precomputed feature vectors + metadata embeddings → 2-layer MLP → destination.

Run `build_centroids.py` before this notebook if `gru_param_config.json`,
`cell_centroids.pt`, and `taxi_id_map.pt` don't exist yet.

In [ ]:
import json
import datetime
import time
from pathlib import Path
from functools import partial

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from tqdm import tqdm

from mlp_model import MLPDestinationModel
from eval import evaluate_model, print_results_table

DATA_DIR = Path('porto_data_bundle')
DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

## 1. Load configs and lookup tables

In [ ]:
with open(DATA_DIR / 'gru_param_config.json') as f:
    cfg = json.load(f)
print('Config:', cfg)

taxi_id_map = torch.load(DATA_DIR / 'taxi_id_map.pt', weights_only=False)
centroids   = torch.load(DATA_DIR / 'cell_centroids.pt', weights_only=False)
print(f'Taxi IDs: {len(taxi_id_map)}  |  Centroids: {len(centroids)}')

## 2. Load region features from heterogeneous graph

In [ ]:
# Only need region node features — no full graph required
graph = torch.load(DATA_DIR / 'hetero_graph.pt', map_location=DEVICE, weights_only=False)
region_feats = graph['region'].x   # [num_regions, 150] — stays on GPU
region_feat_dim = region_feats.shape[1]
print(f'Region feature dim: {region_feat_dim}')
print(f'Region nodes: {region_feats.shape[0]}')

## 3. Dataset and collate_fn

In [ ]:
class ShardDataset(Dataset):
    def __init__(self, shard_paths, desc='Loading'):
        self.examples = []
        for p in tqdm(shard_paths, desc=desc, unit='shard'):
            self.examples.extend(
                torch.load(p, map_location='cpu', weights_only=False)
            )
    def __len__(self):  return len(self.examples)
    def __getitem__(self, idx): return self.examples[idx]


CALL_TYPE_MAP = {'A': 0, 'B': 1, 'C': 2}
DAY_TYPE_MAP  = {'A': 0, 'B': 1, 'C': 2}


def collate_fn(batch, taxi_id_map):
    first_ids = torch.tensor([ex['prefix_region_seq'][0]  for ex in batch], dtype=torch.long)
    last_ids  = torch.tensor([ex['prefix_region_seq'][-1] for ex in batch], dtype=torch.long)

    dest_region = torch.tensor([ex['dest_region'] for ex in batch], dtype=torch.long)
    dest_lat    = torch.tensor([ex['dest_lat']    for ex in batch], dtype=torch.float)
    dest_lon    = torch.tensor([ex['dest_lon']    for ex in batch], dtype=torch.float)

    hours, dows = [], []
    for ex in batch:
        dt = datetime.datetime.fromtimestamp(ex['timestamp'], datetime.timezone.utc)
        hours.append(dt.hour)
        dows.append(dt.weekday())

    return {
        'first_ids'  : first_ids,
        'last_ids'   : last_ids,
        'dest_region': dest_region,
        'dest_lat'   : dest_lat,
        'dest_lon'   : dest_lon,
        'metadata': {
            'call_type': torch.tensor([CALL_TYPE_MAP[ex['call_type']] for ex in batch], dtype=torch.long),
            'taxi_id'  : torch.tensor([taxi_id_map.get(ex['taxi_id'], 0) for ex in batch], dtype=torch.long),
            'day_type' : torch.tensor([DAY_TYPE_MAP[ex['day_type']]   for ex in batch], dtype=torch.long),
            'hour'     : torch.tensor(hours, dtype=torch.long),
            'dow'      : torch.tensor(dows,  dtype=torch.long),
        }
    }

## 4. Build DataLoaders

In [ ]:
BATCH_SIZE = 256
_collate   = partial(collate_fn, taxi_id_map=taxi_id_map)

train_paths = sorted((DATA_DIR / 'supervised_shards' / 'train').glob('train_*.pt'))
val_paths   = sorted((DATA_DIR / 'supervised_shards' / 'val').glob('val_*.pt'))
test_paths  = sorted((DATA_DIR / 'supervised_shards' / 'test').glob('test_*.pt'))

print('Loading val and test shards...')
val_dataset  = ShardDataset(val_paths,  desc='Loading val shards')
test_dataset = ShardDataset(test_paths, desc='Loading test shards')

val_loader  = DataLoader(val_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                         collate_fn=_collate, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         collate_fn=_collate, num_workers=4, pin_memory=True)

print(f'Val: {len(val_dataset):,}  |  Test: {len(test_dataset):,}')

## 5. Training loop (single seed)

In [ ]:
def train_seed(seed: int) -> dict:
    """
    Train MLPDestinationModel with a fixed random seed.
    Returns test set results for the best checkpoint (by val Recall@5).
    """
    torch.manual_seed(seed)
    np.random.seed(seed)

    print(f'\n{"="*60}')
    print(f'SEED {seed}')
    print(f'{"="*60}')

    model = MLPDestinationModel(
        region_feat_dim  = region_feat_dim,
        num_dest_classes = cfg['num_dest_classes'],
        num_taxi_ids     = cfg['num_taxi_ids'],
        hidden_dim       = 256,
        dropout          = 0.3,
    ).to(DEVICE)

    num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Model parameters: {num_params:,}')

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    ckpt_path = f'mlp_best_seed{seed}.pt'

    train_dataset = ShardDataset(train_paths, desc=f'Loading train shards (seed {seed})')
    train_loader  = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True,
        collate_fn=_collate, num_workers=4, pin_memory=True
    )
    print(f'Train: {len(train_dataset):,} examples  |  {len(train_loader):,} batches/epoch\n')

    best_recall5     = 0.0
    patience_counter = 0
    PATIENCE         = 5
    MAX_EPOCHS       = 20
    n_batches        = len(train_loader)

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        total_loss  = 0.0
        epoch_start = time.time()

        pbar = tqdm(train_loader, desc=f'Epoch {epoch:02d}/{MAX_EPOCHS}',
                    unit='batch', leave=True)
        for batch in pbar:
            first_ids = batch['first_ids'].to(DEVICE)
            last_ids  = batch['last_ids'].to(DEVICE)
            dest      = batch['dest_region'].to(DEVICE)
            meta      = {k: v.to(DEVICE) for k, v in batch['metadata'].items()}

            optimizer.zero_grad()
            logits = model(first_ids, last_ids, meta, region_feats)
            loss   = criterion(logits, dest)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            total_loss += loss.item()
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})

        avg_loss   = total_loss / n_batches
        epoch_time = time.time() - epoch_start

        print(f'  Epoch {epoch:02d} | train loss {avg_loss:.4f} | '
              f'time {epoch_time/60:.1f} min | validating...', end=' ', flush=True)

        val_results = evaluate_model(model, val_loader, centroids, DEVICE,
                                     region_feats=region_feats)
        recall5 = val_results['Recall@5']

        print(f'val R@1 {val_results["Recall@1"]:.4f} | '
              f'val R@5 {recall5:.4f} | '
              f'mean H {val_results["Mean Haversine (km)"]:.3f} km', end='')

        if recall5 > best_recall5:
            best_recall5 = recall5
            torch.save(model.state_dict(), ckpt_path)
            patience_counter = 0
            print('  ← best')
        else:
            patience_counter += 1
            print(f'  (patience {patience_counter}/{PATIENCE})')
            if patience_counter >= PATIENCE:
                print(f'  Early stopping at epoch {epoch}.')
                break

    print(f'\nLoading best checkpoint (val R@5 = {best_recall5:.4f}) and evaluating on test...')
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE, weights_only=False))
    test_results = evaluate_model(model, test_loader, centroids, DEVICE,
                                  region_feats=region_feats)
    print(f'[Seed {seed}] Test R@1 {test_results["Recall@1"]:.4f} | '
          f'R@5 {test_results["Recall@5"]:.4f} | '
          f'Mean H {test_results["Mean Haversine (km)"]:.3f} km | '
          f'Med H {test_results["Med Haversine (km)"]:.3f} km')
    return test_results

## 6. Run 3 seeds — report mean ± std

In [ ]:
SEEDS = [0, 1, 2]
seed_results = []

for seed in SEEDS:
    result = train_seed(seed)
    seed_results.append(result)

metrics = ['Recall@1', 'Recall@5', 'Recall@10', 'Mean Haversine (km)', 'Med Haversine (km)']
print('\n=== MLP Baseline — 3-seed summary ===')
for m in metrics:
    vals = [r[m] for r in seed_results]
    print(f'  {m:<25}: {np.mean(vals):.4f} ± {np.std(vals):.4f}')

## 7. Results table

In [ ]:
mlp_avg = {m: float(np.mean([r[m] for r in seed_results])) for m in metrics}
mlp_avg['n'] = seed_results[0]['n']

print_results_table({
    'MLP Baseline': mlp_avg,
})